In [0]:
def count_check(catalog, database, operation_type, table_name, num_diff):
    df = spark.sql(f"DESCRIBE HISTORY {catalog}.{database}.{table_name}")
    df.createOrReplaceTempView("Table_count")

    # ✅ Current count
    current_row = spark.sql(f"""
        SELECT operationMetrics.numOutputRows AS cnt
        FROM Table_count
        WHERE version = (
            SELECT MAX(version)
            FROM Table_count
            WHERE LOWER(TRIM(operation)) = LOWER(TRIM('{operation_type}'))
        )
    """).first()

    final_current_count = int(current_row.cnt) if current_row and current_row.cnt else 0

    # ✅ Previous count
    previous_row = spark.sql(f"""
        SELECT operationMetrics.numOutputRows AS cnt
        FROM Table_count
        WHERE version < (
            SELECT MAX(version)
            FROM Table_count
            WHERE LOWER(TRIM(operation)) = LOWER(TRIM('{operation_type}'))
        )
        ORDER BY version DESC
        LIMIT 1
    """).first()

    final_previous_count = int(previous_row.cnt) if previous_row and previous_row.cnt else 0

    # ✅ Comparison
    if (final_current_count - final_previous_count) < int(num_diff):
        raise Exception(f'Difference is huge in {table_name}')

In [0]:
%sql
GRANT USE CATALOG ON CATALOG projectdev TO `chetanaayal3@gmail.com`;

In [0]:
%sql
ALTER CATALOG projectdev OWNER TO `chetanaayal3@gmail.com`;

In [0]:
%sql
SHOW GRANTS ON CATALOG projectdev;

In [0]:
def insert_test_cases(database,insert_id,insert_test_cases,insert_query,insert_expected_result):
    try:
        spark.sql(
            f"""
            CREATE TABLE IF NOT EXISTS {database}.insert_test_cases (
                id INT,
                test_cases STRING,
                test_query STRING,
                expected_result int
            )"""
        )
        spark.sql(
            f"""
            insert into {database}.insert_test_cases(id,test_cases,test_query,expected_result) values
            ({insert_id},'{insert_test_cases}','{insert_query}',{insert_expected_result})
            """
        )
    except Exception as e:
        print(e)
     


In [0]:
def execute_test_case(database:str):
    df = spark.sql(f"SELECT * FROM {database}.insert_test_cases").collect()
    for i in df:
        original_result = spark.sql(f"""{i.test_query}""").collect()
        if(len(original_result) == i.expected_result):
            print("Test case passed")
        else:
            raise Exception(f"Test case{i.test_query} is failed")